# 07 - PySpark core patterns

In [1]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "de_helpers.py").exists())
sys.path.insert(0, str(ROOT / "src"))

# Databricks Connect session via Databricks CLI authentication
from de_helpers import (
    q, q1, spark_q, assert_grain, tables, columns, explain, display, fs_ls,
    get_spark, get_dbutils, spark_read, plan_string, has_plan_operator
)
from pyspark.sql import functions as F, Window

spark = get_spark()
dbutils = get_dbutils()


**Ten Spark drills on the data you already queried in SQL** - `silver/claims.parquet` (11,981 claims) + `providers` (220).
Each drill is a plan block, a `# YOUR TURN` stub, then the solution. Every output is deliberately tiny.
Spark makes you own what SQL hides: shuffle boundaries, partition counts, and what runs in the JVM vs in Python.
Answer the interview question, then be able to say why the plan looks the way it does.

In [5]:
import os
os.environ["PYSPARK_PYTHON"] = os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable  # venv 3.12; bare python3 here is 3.13
from de_helpers import get_spark, spark_read, has_plan_operator
from pyspark.sql import functions as F, Window
from pyspark.sql.types import StringType

spark = get_spark("07-pyspark-core")
claims = spark_read(spark, "silver", "claims")
providers = spark_read(spark, "silver", "providers")
print("spark", spark.version, "| claims", claims.count(), "| providers", providers.count())

spark 4.2.0 | claims 11981 | providers 220


**1 - Read & inspect.** `spark_read(spark, "silver", "claims")`.
- Parquet is columnar and typed: Spark reads only the projected columns and never parses the ones it skips.
- The same bytes as CSV would be scanned in full, every column arriving as a string that still needs casting.
**Spark note:** projection pushdown is free - `select` the 4 columns you need and the other 11 are never read.

In [7]:
# 1. Check session type (Spark Connect = Databricks remote)
print("Session class:", type(spark))
# -> <class 'pyspark.sql.connect.session.SparkSession'>

# 2. Check current catalog and schema on Databricks
spark.sql("SELECT current_catalog(), current_schema(), current_user()").show(truncate=False)

# Check workspace URL
print("Connected to:", spark.client._builder.host)



Session class: <class 'pyspark.sql.connect.session.SparkSession'>
+-----------------+----------------+----------------------+
|current_catalog()|current_schema()|current_user()        |
+-----------------+----------------+----------------------+
|workspace        |healthcare      |iraonfridays@gmail.com|
+-----------------+----------------+----------------------+

Connected to: dbc-2944edfb-cd25.cloud.databricks.com


In [10]:
# YOUR TURN -> print the schema, the row count, a 3-row peek
# expected columns: claim_id, status, paid_amount, claim_date

print("columns:", len(claims.columns))
claims.printSchema()
print("rows:", claims.count())
claims.select("claim_id","status", "paid_amount", "claim_date").show(3, truncate=False)

columns: 15
root
 |-- claim_id: string (nullable = true)
 |-- member_id: string (nullable = true)
 |-- provider_id: string (nullable = true)
 |-- claim_date: date (nullable = true)
 |-- submitted_date: date (nullable = true)
 |-- updated_at: timestamp_ntz (nullable = true)
 |-- status: string (nullable = true)
 |-- claim_type: string (nullable = true)
 |-- primary_diagnosis_code: string (nullable = true)
 |-- billed_amount: decimal(12,2) (nullable = true)
 |-- paid_amount: decimal(12,2) (nullable = true)
 |-- ingested_at: timestamp_ntz (nullable = true)
 |-- source_system: string (nullable = true)
 |-- _ingest_batch: string (nullable = true)
 |-- _loaded_at: timestamp_ntz (nullable = true)

rows: 11981
+----------+--------+-----------+----------+
|claim_id  |status  |paid_amount|claim_date|
+----------+--------+-----------+----------+
|CLM0000002|Approved|7150.71    |2024-01-24|
|CLM0000005|Approved|810.02     |2024-01-01|
|CLM0000028|Approved|1597.07    |2024-01-09|
+----------+------

In [8]:
print("columns:", len(claims.columns))
claims.printSchema()
print("rows:", claims.count())
claims.select("claim_id", "status", "paid_amount", "claim_date").show(3, truncate=False)

columns: 15
root
 |-- claim_id: string (nullable = true)
 |-- member_id: string (nullable = true)
 |-- provider_id: string (nullable = true)
 |-- claim_date: date (nullable = true)
 |-- submitted_date: date (nullable = true)
 |-- updated_at: timestamp_ntz (nullable = true)
 |-- status: string (nullable = true)
 |-- claim_type: string (nullable = true)
 |-- primary_diagnosis_code: string (nullable = true)
 |-- billed_amount: decimal(12,2) (nullable = true)
 |-- paid_amount: decimal(12,2) (nullable = true)
 |-- ingested_at: timestamp_ntz (nullable = true)
 |-- source_system: string (nullable = true)
 |-- _ingest_batch: string (nullable = true)
 |-- _loaded_at: timestamp_ntz (nullable = true)

rows: 11981
+----------+--------+-----------+----------+
|claim_id  |status  |paid_amount|claim_date|
+----------+--------+-----------+----------+
|CLM0000002|Approved|7150.71    |2024-01-24|
|CLM0000005|Approved|810.02     |2024-01-01|
|CLM0000028|Approved|1597.07    |2024-01-09|
+----------+------

**2 - Aggregation.** Total paid and claim count per provider.
- `groupBy().agg()` and `spark.sql()` compile to the same plan - answer in whichever the interviewer asks for.
- `SUM` skips the NULL `paid_amount` of unadjudicated claims; `COUNT(*)` still counts those rows.
**Spark note:** `groupBy` shuffles by key; 220 groups is trivial, but size `spark.sql.shuffle.partitions` to the data, not to the default.

In [25]:
# YOUR TURN -> per provider_id: total_paid, claim_count (top 5 by total_paid)
by_provider = (
                claims
                .groupBy("provider_id")
                .agg(F.round(F.sum("paid_amount"), 2).alias("total_paid") , F.count("*").alias("claim_count"))

                )

by_provider.orderBy(F.desc("total_paid")).show(5,truncate=False)

claims.createOrReplaceTempView("claims_v")
spark.sql('''
    select 
        provider_id,
        round(sum(paid_amount), 2) as total_paid,
        count(*) as claim_count
        from claims_v
        group by provider_id
        order by total_paid desc 
        limit 5;
''').show(truncate=False)

+-----------+----------+-----------+
|provider_id|total_paid|claim_count|
+-----------+----------+-----------+
|PRV0151    |1430804.55|65         |
|PRV0164    |1217394.15|69         |
|PRV0103    |1027851.72|66         |
|PRV0010    |1014922.90|53         |
|PRV0041    |1005744.56|49         |
+-----------+----------+-----------+
only showing top 5 rows
+-----------+----------+-----------+
|provider_id|total_paid|claim_count|
+-----------+----------+-----------+
|PRV0151    |1430804.55|65         |
|PRV0164    |1217394.15|69         |
|PRV0103    |1027851.72|66         |
|PRV0010    |1014922.90|53         |
|PRV0041    |1005744.56|49         |
+-----------+----------+-----------+



In [16]:
by_provider = (
               claims
               .groupBy("provider_id")
               .agg(F.round(F.sum("paid_amount"), 2).alias("total_paid"),
                    F.count("*").alias("claim_count"))
               )
by_provider.orderBy(F.desc("total_paid")).show(5, truncate=False)

claims.createOrReplaceTempView("claims_v")
spark.sql('''SELECT 
               provider_id, 
               ROUND(SUM(paid_amount), 2) AS total_paid, 
               COUNT(*) AS claim_count
             FROM claims_v 
             GROUP BY provider_id 
             ORDER BY total_paid DESC LIMIT 5
             '''
          ).show(truncate=False)
print("grand total paid:", by_provider.agg(F.round(F.sum("total_paid"), 2)).first()[0])

+-----------+----------+-----------+
|provider_id|total_paid|claim_count|
+-----------+----------+-----------+
|PRV0151    |1430804.55|65         |
|PRV0164    |1217394.15|69         |
|PRV0103    |1027851.72|66         |
|PRV0010    |1014922.90|53         |
|PRV0041    |1005744.56|49         |
+-----------+----------+-----------+
only showing top 5 rows
+-----------+----------+-----------+
|provider_id|total_paid|claim_count|
+-----------+----------+-----------+
|PRV0151    |1430804.55|65         |
|PRV0164    |1217394.15|69         |
|PRV0103    |1027851.72|66         |
|PRV0010    |1014922.90|53         |
|PRV0041    |1005744.56|49         |
+-----------+----------+-----------+

grand total paid: 36897727.38


**3 - Join + broadcast.** claims x providers (220 rows) to get state and specialty.
- `F.broadcast(small)` ships the whole small side to every executor, so the big side is never shuffled.
- Appropriate when one side genuinely fits in memory and is reused; wrong for two large tables (driver OOM, no gain).
**Spark note:** `spark.sql.autoBroadcastJoinThreshold` (10 MB) does this automatically - the hint wins when the optimizer's stats are missing.

In [32]:

# YOUR TURN -> state, specialty, total_paid through a broadcast join (top 5)
by_segment = (
                claims.join(F.broadcast(providers.select("provider_id","state","specialty")), on="provider_id")
                .groupBy("state","specialty")
                .agg(F.round(F.sum("paid_amount"),2).alias("total_paid"))
            )
by_segment.orderBy(F.desc("total_paid")).show(5,truncate = False)

+-----+-----------+----------+
|state|specialty  |total_paid|
+-----+-----------+----------+
|CA   |Orthopedics|2887877.65|
|IL   |Orthopedics|2357907.53|
|NY   |Orthopedics|2167268.35|
|FL   |OB/GYN     |1409859.15|
|FL   |Oncology   |1165149.96|
+-----+-----------+----------+
only showing top 5 rows


In [ ]:
by_segment = (claims.join(F.broadcast(providers.select("provider_id", "state", "specialty")), "provider_id")
              .groupBy("state", "specialty")
              .agg(F.round(F.sum("paid_amount"), 2).alias("total_paid"))
              )
by_segment.orderBy(F.desc("total_paid")).show(5, truncate=False)
print("broadcast join in the plan:", has_plan_operator(by_segment, "BroadcastHashJoin"))

+-----+-----------+----------+
|state|specialty  |total_paid|
+-----+-----------+----------+
|CA   |Orthopedics|2887877.65|
|IL   |Orthopedics|2357907.53|
|NY   |Orthopedics|2167268.35|
|FL   |OB/GYN     |1409859.15|
|FL   |Oncology   |1165149.96|
+-----+-----------+----------+
only showing top 5 rows
broadcast join in the plan: True


In [14]:
by_segment.explain(mode="formatted")


== Physical Plan ==
AdaptiveSparkPlan (16)
+- == Initial Plan ==
   PhotonResultStage (15)
   +- PhotonColumnarToRow (14)
      +- PhotonGroupingAgg (13)
         +- PhotonShuffleExchangeSource (12)
            +- PhotonShuffleMapStage (11)
               +- PhotonShuffleExchangeSink (10)
                  +- PhotonGroupingAgg (9)
                     +- PhotonProject (8)
                        +- PhotonBroadcastHashJoin Inner (7)
                           :- PhotonScan parquet  (1)
                           +- PhotonShuffleExchangeSource (6)
                              +- PhotonShuffleMapStage (5)
                                 +- PhotonShuffleExchangeSink (4)
                                    +- PhotonProject (3)
                                       +- PhotonScan parquet  (2)


(1) PhotonScan parquet 
Output [2]: [provider_id#10060, paid_amount#10068]
Location: InMemoryFileIndex [dbfs:/Volumes/workspace/iraonfridays/de_prep/silver/claims.parquet]
OptionalDataFilters: [hash

The filter **IS running BEFORE the join!** 

Your intuition that filtering should happen before the join is 100% correct—and that is exactly what Spark did.

Here is why it looks confusing and how to read the tree:

---

### 1. The Rule: Spark Plans Run from BOTTOM to TOP

Spark prints plans as an inverted tree where the **leaves (data sources) are at the bottom** and the **final result is at the top**.

Execution flows **from the innermost indentation (bottom) upwards**:

```text
Final Result (Top)
       ▲
   Aggregate                   <-- Step 4: GroupBy & Sum
       ▲
     Join                      <-- Step 3: Join executes HERE
    /    \
 Left    Right
  │        │
Project  Project               <-- Step 2: Prune unused columns
  │        │
Filter   Filter                <-- Step 1: FILTER RUNS FIRST!
  │        │
Relation Relation              <-- Step 0: Read parquet from disk (Leaves)
```

---

### 2. How to Read the Join Branches (`:-` and `+-`)

In Spark's plan, a `Join` has two branches:
* **`:-`** = The **Left child** (the `claims` table)
* **`+-`** = The **Right child** (the `providers` table)

Look at the indentation under your `Join`:

```text
+- Join Inner, (provider_id#... = provider_id#...), rightHint=(strategy=broadcast)
   :- Project [provider_id#..., paid_amount#...]          <-- 3. Keep only 2 columns
   :  +- Filter isnotnull(provider_id#...)                <-- 2. FILTER RUNS HERE (Before join!)
   :     +- Relation [...] parquet                        <-- 1. Read claims from disk
```

Because `Filter isnotnull` is indented **inside** the left child of `Join`, Spark scans the parquet file, **filters it first**, drops unused columns, and **then feeds only the clean rows up into the `Join`**.

---

### 3. A Cool Catalyst Optimizer Detail (Interview Pro-Tip ⭐)

Notice this line:
```text
+- Filter isnotnull(provider_id#10060)
```
**You never wrote `isnotnull(provider_id)` in your PySpark code!** Where did it come from?

Spark’s **Catalyst Optimizer** applied a rule called **Infer Filters from Constraints**:
* Catalyst knows that in an `INNER JOIN on provider_id = provider_id`, `NULL = NULL` can never evaluate to `TRUE`.
* Therefore, it automatically generated and pushed down `isnotnull(provider_id)` straight to the parquet scan layer to discard useless rows **before** doing any join work!

---

### Quick Cheat Sheet for Reading Plans

| Symbol / Operator | What it means |
| :--- | :--- |
| **`Relation ... parquet`** | Reading the raw file from storage (happens first at the bottom). |
| **`Filter ...`** | Predicate Pushdown (filtering rows as early as possible). |
| **`Project [...]`** | Column Pruning (dropping unused columns to save memory). |
| **`Join Inner`** | The join itself (takes the outputs of the two branches below it). |
| **`Aggregate`** | Group by and aggregate calculations (happens towards the top). |

In [15]:
by_segment.explain(mode="extended")


== Parsed Logical Plan ==
'Aggregate ['state, 'specialty], ['state, 'specialty, 'round('sum('paid_amount), 2) AS total_paid#10081]
+- 'Join UsingJoin(Inner, [provider_id])
   :- Relation [claim_id#10058,member_id#10059,provider_id#10060,claim_date#10061,submitted_date#10062,updated_at#10063,status#10064,claim_type#10065,primary_diagnosis_code#10066,billed_amount#10067,paid_amount#10068,ingested_at#10069,source_system#10070,_ingest_batch#10071,_loaded_at#10072] parquet
   +- 'UnresolvedHint broadcast
      +- 'Project ['provider_id, 'state, 'specialty]
         +- Relation [provider_id#10074,npi#10075L,provider_name#10076,specialty#10077,state#10078,network_status#10079,updated_at#10080] parquet

== Analyzed Logical Plan ==
state: string, specialty: string, total_paid: decimal(23,2)
Aggregate [state#10078, specialty#10077], [state#10078, specialty#10077, round(sum(paid_amount#10068), 2) AS total_paid#10081]
+- Project [provider_id#10060, claim_id#10058, member_id#10059, claim_date#10061

**4 - Anti-join.** Members with no claims.
- `left_anti` is SQL `NOT EXISTS`: keep left rows with no match, and it can never duplicate the left side.
- `left_join ... WHERE right IS NULL` does fan out when the right side is not unique - the classic bug.
**Spark note:** both sides are shuffled unless the right side is broadcast; the anti-join also pushes the key projection down.

In [ ]:
# YOUR TURN -> members with zero claims: member_id, first_name, last_name + the count

In [37]:
# 1. Load tables
members = spark.table("members")
claims = spark.table("claims")

# 2. Left anti-join directly on member_id (NO distinct needed!)
no_claims = members.join(claims, on="member_id", how="left_anti")

# 3. Print metrics and preview
print(f"Total members: {members.count():,} | Members with zero claims: {no_claims.count():,}")
no_claims.select("member_id", "first_name", "last_name").show(5, truncate=False)


Total members: 1,200 | Members with zero claims: 5
+---------+----------+---------+
|member_id|first_name|last_name|
+---------+----------+---------+
|MBR01169 |Charles   |Lee      |
|MBR00544 |Hiro      |Miller   |
|MBR00712 |Karen     |Taylor   |
|MBR00890 |Nancy     |Thompson |
|MBR00909 |Ashley    |Davis    |
+---------+----------+---------+



In [38]:
spark.sql('''
    SELECT COUNT(*) AS no_claims
    FROM members
    LEFT ANTI JOIN claims USING (member_id)
''').show()


+---------+
|no_claims|
+---------+
|        5|
+---------+



Finding records in Table A that do **not** exist in Table B is known as an **Anti-Join**. 

Here are the **5 different ways** to write this in modern Databricks SQL, PySpark, and DuckDB, ranked from most recommended to least recommended:

---

### 1. `LEFT ANTI JOIN` (Modern Best Practice ⭐)
Both **Databricks SQL** and **DuckDB** support native `LEFT ANTI JOIN`. It tells the optimizer directly what you want without workarounds:

#### Databricks / DuckDB SQL:
```sql
SELECT COUNT(*) AS no_claims
FROM members_v m
LEFT ANTI JOIN claims_v c
    ON m.member_id = c.member_id
```

#### PySpark DataFrame API:
```python
no_claims = members.join(claims, on="member_id", how="left_anti").count()
print("no_claims:", no_claims)
```
* **Why it's best:** Cleanest syntax, 100% null-safe, and Spark's Catalyst engine immediately plans a `BroadcastHashJoin (LeftAnti)` or `SortMergeJoin (LeftAnti)` without unnecessary overhead.

---

### 2. `LEFT JOIN ... WHERE right.key IS NULL` (Traditional ANSI SQL)
The classic approach that works in every relational database engine ever built:

#### SQL:
```sql
SELECT COUNT(*) AS no_claims
FROM members_v m
LEFT JOIN claims_v c
    ON m.member_id = c.member_id
WHERE c.member_id IS NULL
```

#### PySpark DataFrame API:
```python
no_claims = (
    members
    .join(claims.select("member_id", "claim_id"), on="member_id", how="left")
    .filter(F.col("claim_id").isNull())
    .count()
)
```
* **Under the hood:** In Databricks / Spark 3.x+, the Catalyst optimizer detects the `LEFT JOIN + IS NULL` pattern and automatically rewrites it to an Anti-Join physical plan.

---

### 3. `WHERE NOT EXISTS` (Correlated Subquery)
What you originally wrote. Supported everywhere:

#### SQL:
```sql
SELECT COUNT(*) AS no_claims
FROM members_v m
WHERE NOT EXISTS (
    SELECT 1 
    FROM claims_v c 
    WHERE c.member_id = m.member_id
)
```
* **Under the hood:** Modern engines (Databricks, DuckDB) **decorrelate** this subquery during the Logical Optimization phase and execute it as an Anti-Join.

---

### 4. `EXCEPT` / `subtract()` (Set Difference)
Focuses strictly on the distinct primary keys:

#### SQL:
```sql
SELECT COUNT(*) AS no_claims
FROM (
    SELECT member_id FROM members_v
    EXCEPT
    SELECT member_id FROM claims_v
)
```

#### PySpark DataFrame API:
```python
no_claims = (
    members.select("member_id")
    .subtract(claims.select("member_id"))
    .count()
)
```
* **Note:** `EXCEPT` inherently deduplicates keys (`DISTINCT`), which adds a shuffle step if keys aren't already unique.

---

### 5. `WHERE NOT IN` (⚠️ The Classic Interview Trap)

```sql
-- ⚠️ DANGEROUS: If claims_v has even ONE NULL member_id, this returns 0 rows!
SELECT COUNT(*) AS no_claims
FROM members_v
WHERE member_id NOT IN (
    SELECT member_id FROM claims_v WHERE member_id IS NOT NULL  -- NULL filter required!
)
```

#### 🚨 The Three-Valued Logic Trap (Favorite DE Interview Question):
If `claims_v` contains even a single `NULL` in `member_id`:
1. `member_id NOT IN (1, 2, NULL)` evaluates to `NOT (member_id = 1 OR member_id = 2 OR member_id = NULL)`.
2. Any comparison with `NULL` yields `UNKNOWN`.
3. `NOT (UNKNOWN)` is still `UNKNOWN`.
4. The `WHERE` clause rejects every single row, **silently returning 0 rows!**

---

### Summary Comparison Table for Interviews

| Method | Null-Safe? | Code Cleanliness | Recommendation |
| :--- | :--- | :--- | :--- |
| **`LEFT ANTI JOIN`** | ✅ Yes | Cleanest | **Use this in Modern Databricks / DuckDB** |
| **`LEFT JOIN ... IS NULL`** | ✅ Yes | Standard | Standard fallback for legacy SQL |
| **`WHERE NOT EXISTS`** | ✅ Yes | Good | Good for complex multi-condition checks |
| **`EXCEPT` / `subtract`** | ✅ Yes | Verbose | Great when comparing distinct ID lists |
| **`WHERE NOT IN`** | ❌ **Dangerous** | Risky | **Avoid** unless explicitly guarded with `WHERE col IS NOT NULL` |

**5 - Deduplicate claim versions.** `bronze.claims` = 12,358 rows with 354 duplicated `claim_id` (source versions + a replay batch).
- Keep the newest `updated_at` per claim with `row_number()`, and break ties on `_ingest_batch` so the pick is deterministic.
- Expect 11,996 surviving keys - the same result as the SQL `ROW_NUMBER` drill.
**Spark note:** `row_number` over `partitionBy("claim_id")` is a shuffle, but into many small partitions - the cheap direction.

In [34]:
# YOUR TURN -> one row per claim_id: claim_id, updated_at, _ingest_batch (11,996 rows)

In [35]:
bronze = spark_read(spark, "bronze", "claims")
w_latest = Window.partitionBy("claim_id").orderBy(F.col("updated_at").desc(), F.col("_ingest_batch").desc())
latest = bronze.withColumn("rn", F.row_number().over(w_latest)).filter(F.col("rn") == 1).drop("rn")
print("bronze rows:", bronze.count(), "| distinct claim_id:", latest.count())
latest.select("claim_id", "updated_at", "_ingest_batch").show(3, truncate=False)

bronze rows: 12358 | distinct claim_id: 11996


KeyboardInterrupt: 

**6 - Top-N per group.** Top 3 providers per state by total paid.
- Aggregate first, then `rank()` over `Window.partitionBy("state").orderBy(F.desc("total_paid"))`.
- `rank()` keeps ties, so a state can return 4 rows; `row_number()` forces exactly 3.
**Spark note:** the window rides on top of the aggregation shuffle, so there is one shuffle, not two.

In [ ]:
# YOUR TURN -> state, provider_id, total_paid, rnk where rnk <= 3

In [ ]:
totals = (claims.join(F.broadcast(providers.select("provider_id", "state")), "provider_id")
          .groupBy("state", "provider_id")
          .agg(F.round(F.sum("paid_amount"), 2).alias("total_paid")))
by_state = Window.partitionBy("state").orderBy(F.col("total_paid").desc())
top3 = totals.withColumn("rnk", F.rank().over(by_state)).filter(F.col("rnk") <= 3)
print("rows returned (20 states, ties can add):", top3.count())
top3.orderBy("state", "rnk").show(6, truncate=False)

**7 - LAG.** Month-over-month change in total paid.
- `lag()` needs an ordered window, and `Window.orderBy(...)` with no `partitionBy` collapses everything into ONE partition.
- So aggregate to the 18 monthly rows first and window that tiny result - never window the raw claims.
**Spark note:** a single-partition window puts all rows in one task; after the `groupBy` the result is already one partition, so it is free here.

In [ ]:
# YOUR TURN -> month, total_paid, prev_paid, mom_change (18 rows)

In [ ]:
monthly = (claims.groupBy(F.date_format("claim_date", "yyyy-MM").alias("month"))
           .agg(F.round(F.sum("paid_amount"), 2).alias("total_paid")))
mom = (monthly.withColumn("prev_paid", F.lag("total_paid").over(Window.orderBy("month")))
       .withColumn("mom_change", F.round(F.col("total_paid") - F.col("prev_paid"), 2)))
try:
    num_parts = monthly.rdd.getNumPartitions()
except Exception:
    num_parts = "remote (adaptive)"
print("rows into the window stage:", mom.count(), "| partitions:", num_parts)
mom.show(4, truncate=False)

m:\databricks\databricks-wanderbricks\de-interview-prep\.venv\Lib\site-packages\pyspark\sql\connect\expressions.py:1153: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


rows into the window stage: 18 | partitions: remote (adaptive)
+-------+----------+----------+----------+
|month  |total_paid|prev_paid |mom_change|
+-------+----------+----------+----------+
|2024-01|1391910.29|NULL      |NULL      |
|2024-02|1593433.87|1391910.29|201523.58 |
|2024-03|1589444.78|1593433.87|-3989.09  |
|2024-04|1457570.66|1589444.78|-131874.12|
+-------+----------+----------+----------+
only showing top 4 rows


**8 - Conditional aggregation.** Approval rate per provider.
- `SUM(CASE WHEN status = 'Approved' THEN 1 ELSE 0 END) / COUNT(*)` is exactly `F.sum(F.when(...).otherwise(0)) / F.count("*")`.
- Guard the denominator: `F.when(count == 0, None).otherwise(count)` is Spark's `NULLIF(count, 0)` - no divide-by-zero.
**Spark note:** one pass, no extra shuffle; the CASE is folded into the same aggregation as the `COUNT`.

In [ ]:
# YOUR TURN -> provider_id, claims, approved, approval_rate (top 5 by rate)

In [ ]:
df = q('''
WITH provider_stats AS (
    SELECT
        provider_id,
        COUNT(*)                      AS total_claims,
        COUNT_IF(status = 'Approved') AS approved
    FROM claims
    GROUP BY provider_id
)
SELECT
    provider_id,
    total_claims,
    approved,
    ROUND(approved * 1.0 / NULLIF(total_claims, 0), 4) AS approval_rate
FROM provider_stats
ORDER BY approval_rate DESC, total_claims DESC
LIMIT 5;
''')
df

,provider_id,total_claims,approved,approval_rate
0,PRV0121,59,51.00,0.86
1,PRV0107,61,52.00,0.85
2,PRV0137,54,46.00,0.85
3,PRV0173,54,46.00,0.85
4,PRV0077,47,40.00,0.85


In [ ]:
denom = F.when(F.col("claims") == 0, F.lit(None)).otherwise(F.col("claims"))   # NULLIF(claims, 0)
rate = (claims.groupBy("provider_id")
        .agg(F.count("*").alias("claims"),
             F.sum(F.when(F.col("status") == "Approved", 1).otherwise(0)).alias("approved"))
        .withColumn("approval_rate", F.round(F.col("approved") / denom, 4)))
rate.orderBy(F.desc("approval_rate"), F.desc("claims")).show(5, truncate=False)

+-----------+------+--------+-------------+
|provider_id|claims|approved|approval_rate|
+-----------+------+--------+-------------+
|PRV0121    |59    |51      |0.8644       |
|PRV0107    |61    |52      |0.8525       |
|PRV0137    |54    |46      |0.8519       |
|PRV0173    |54    |46      |0.8519       |
|PRV0077    |47    |40      |0.8511       |
+-----------+------+--------+-------------+
only showing top 5 rows


In [ ]:
from pyspark.sql import functions as F

rate = (
    claims
    .groupBy("provider_id")
    .agg(
        F.count("*").alias("claims"),
        F.count_if(F.col("status") == "Approved").alias("approved")      # Replaces F.sum(F.when(...))
    )
    .withColumn(
        "approval_rate",
        F.round(F.col("approved") / F.nullif(F.col("claims"), F.lit(0)), 4)  # Replaces denom F.when(...)
    )
    .orderBy(F.desc("approval_rate"), F.desc("claims"))
)

rate.show(5, truncate=False)


+-----------+------+--------+-------------+
|provider_id|claims|approved|approval_rate|
+-----------+------+--------+-------------+
|PRV0121    |59    |51      |0.8644       |
|PRV0107    |61    |52      |0.8525       |
|PRV0137    |54    |46      |0.8519       |
|PRV0173    |54    |46      |0.8519       |
|PRV0077    |47    |40      |0.8511       |
+-----------+------+--------+-------------+
only showing top 5 rows


**9 - Nested JSON.** Land 3 nested records in `data/scratch/nested_demo.json`, read them, flatten them.
- Struct access is `F.col("patient.state")`; `F.explode("services")` turns the array into one row per service line.
- Before: nested struct/array. After: flat scalars you can join and aggregate.
**Spark note:** `explode` silently drops rows with an empty array (claim C3) - use `explode_outer` when the parent row must survive.

In [ ]:
# YOUR TURN -> claim_id, member_id, state, procedure_code, amount (one row per service line)

In [ ]:
import json
scratch = pathlib.Path(ROOT) / "data" / "scratch"
scratch.mkdir(parents=True, exist_ok=True)
demo = scratch / "nested_demo.json"
records = [
    {"claim_id": "C1", "patient": {"member_id": "M1", "state": "CA"},
     "services": [{"procedure_code": "99213", "amount": 120.0}, {"procedure_code": "80053", "amount": 45.5}]},
    {"claim_id": "C2", "patient": {"member_id": "M2", "state": "TX"},
     "services": [{"procedure_code": "99214", "amount": 200.0}]},
    {"claim_id": "C3", "patient": {"member_id": "M3", "state": "NY"}, "services": []},
]
demo.write_text("\n".join(json.dumps(r) for r in records))

is_remote = spark.__module__.startswith("pyspark.sql.connect") or hasattr(spark, "client")
nested = spark.createDataFrame(records) if is_remote else spark.read.json(str(demo))
flat = (nested.select("claim_id", "patient", F.explode("services").alias("svc"))
        .select("claim_id",
                F.col("patient.member_id").alias("member_id"),
                F.col("patient.state").alias("state"),
                F.col("svc.procedure_code").alias("procedure_code"),
                F.col("svc.amount").alias("amount")))
print("BEFORE:"); nested.printSchema()
print("AFTER:"); flat.printSchema()
print("nested rows:", nested.count(), "-> flat service lines:", flat.count())
flat.show(truncate=False)

BEFORE:
root
 |-- claim_id: string (nullable = true)
 |-- patient: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- services: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: string
 |    |    |-- value: string (valueContainsNull = true)

AFTER:
root
 |-- claim_id: string (nullable = true)
 |-- member_id: string (nullable = true)
 |-- state: string (nullable = true)
 |-- procedure_code: string (nullable = true)
 |-- amount: string (nullable = true)

nested rows: 3 -> flat service lines: 3
+--------+---------+-----+--------------+------+
|claim_id|member_id|state|procedure_code|amount|
+--------+---------+-----+--------------+------+
|C1      |M1       |CA   |99213         |120.0 |
|C1      |M1       |CA   |80053         |45.5  |
|C2      |M2       |TX   |99214         |200.0 |
+--------+---------+-----+--------------+------+



**10 - Native function vs Python UDF.** Uppercase `status` both ways.
- The UDF ships every batch to a Python worker and back, and Catalyst cannot see inside the lambda to optimise it.
- `F.upper` is a JVM expression: it is code-generated, fused into the scan, and needs no Python on the executor.
**Spark note:** the plan below is the tell - `BatchEvalPython` appears for the UDF and not for the native call.

In [ ]:
# YOUR TURN -> status_raw, status_native, status_udf for 3 claims


In [ ]:
upper_udf = F.udf(lambda s: s.upper() if s else None, StringType())
try:
    both = claims.select(F.col("status").alias("status_raw"),
                         F.upper("status").alias("status_native"),
                         upper_udf("status").alias("status_udf"))
    both.show(3, truncate=False)
    print("native call uses a Python worker:", has_plan_operator(both.select("status_native"), "BatchEvalPython"))
    print("udf call    uses a Python worker:", has_plan_operator(both.select("status_udf"), "BatchEvalPython"))
except Exception as e:
    print("Python worker UDF notice:", type(e).__name__)
    claims.select(F.col("status").alias("status_raw"), F.upper("status").alias("status_native")).show(3, truncate=False)
    print("native call uses a Python worker:", has_plan_operator(claims.select(F.upper("status")), "BatchEvalPython"))

The difference between **`F.upper` (Native Spark function)** and a **Python UDF (`F.udf`)** is one of the **most frequently asked questions in PySpark / Data Engineering interviews**.

Here is what happens under the hood and why it makes a **10x to 100x performance difference**:

---

### 1. How They Execute Under the Hood

```
[Native F.upper]:
Data in JVM Memory ──► Runs compiled C++/Java code directly ──► Result
(Zero overhead, runs directly on Spark's Tungsten engine)


[Python UDF]:
Data in JVM Memory
       │
       ▼ (1. Serialize data via Py4J / Pickle)
    OS Pipe / Socket
       │
       ▼ (2. Send to separate Python Worker Process)
  python.exe / GIL
       │ (3. Deserialize into Python string & run lambda)
       ▼ (4. Serialize return value back)
    OS Pipe / Socket
       │
       ▼ (5. Deserialize back into JVM Memory)
Result in JVM
```

#### A. `F.upper` (Native Catalyst / Tungsten Function)
* Runs **entirely inside the JVM** (and on Databricks, inside the ultra-fast C++ **Photon** engine).
* Operates directly on Spark’s binary in-memory format (**Tungsten**).
* **Whole-Stage Code Generation:** Catalyst compiles the operation directly into raw bytecode with vectorized CPU instructions.
* **Zero serialization:** Data never leaves the JVM.

#### B. Python UDF (`F.udf(lambda ...)`)
* Spark's executors run in the **JVM**. Python code cannot run directly in Java.
* Therefore, for every batch of rows, Spark must spawn a separate **Python Worker process** (`python.exe`).
* Data has to be **serialized** (Pickled) from Java, pushed through a local socket/pipe to Python, executed row-by-row in the Python interpreter, **serialized back**, and deserialized into the JVM.
* In the physical execution plan, this appears as the expensive operator: **`BatchEvalPython`**.

---

### 2. Comparison Table

| Aspect | `F.upper` (Native) | Python UDF (`F.udf`) |
| :--- | :--- | :--- |
| **Execution Engine** | JVM / C++ Photon engine | Separate Python process (`python.exe`) |
| **Speed** | ⚡ **Blazing fast (Baseline)** | 🐢 **10x to 50x slower** |
| **Serialization Overhead** | **Zero** | **Very high** (Pickle IPC serialization) |
| **Catalyst Optimization** | Full optimization & Whole-Stage CodeGen | **Black box** (Catalyst cannot see inside Python code) |
| **Physical Plan Operator** | `Project [upper(status#...)]` | `BatchEvalPython [<lambda>(status#...)]` |

---

### 3. How to See This in Your Notebook

In your code snippet, this exact difference is measured by:
```python
# Returns False (runs purely in JVM, no Python worker involved!)
has_plan_operator(both.select("status_native"), "BatchEvalPython")

# Returns True (forced Spark to spin up a Python worker and serialize rows!)
has_plan_operator(both.select("status_udf"), "BatchEvalPython")
```

---

### 4. The Senior DE Hierarchy of Functions (Interview Best Practice)

When transforming columns in PySpark, always follow this priority order:

1. 🥇 **Native `pyspark.sql.functions` (`F.upper`, `F.coalesce`, `F.concat`):** Always use first.
2. 🥈 **`F.expr("...")`:** If a complex SQL function exists (e.g., regex, date math), use inline SQL expressions.
3. 🥉 **Pandas UDF (`@pandas_udf` / Arrow-optimized):** If custom Python code is truly required, use Vectorized Pandas UDFs—they transfer Apache Arrow batches instead of row-by-row pickling.
4. ❌ **Standard Python UDF (`F.udf(lambda ...)`):** **Last resort only** (e.g., when calling an external proprietary Python library that has no SQL/JVM equivalent).

**Interview checklist:** projection pushdown, one shuffle per aggregation, broadcast the small side, dedupe before you join, aggregate before you window.

In [ ]:
spark.stop()
print("spark stopped - run the next notebook in a fresh kernel")